In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
pip install pandas numpy scikit-learn catboost shap joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from catboost import CatBoostClassifier, Pool
import joblib


df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/revenue_recovery_dataset.csv')

# Scope to failed payments with insufficient_funds only (current slice)
df = df[(df['failed_payment'] == True) & (df['decline_reason'] == 'insufficient_funds')].copy()
print(f"Rows in scope: {len(df)}")
print(df['recovered'].value_counts(normalize=True))


# 2. Feature engineering
df['prior_retry_success_rate'] = df['prior_retry_success_rate'].fillna(-1)  # -1 flags cold start
df['has_history'] = (df['prior_retry_success_rate'] != -1).astype(int)
df['is_near_payday'] = df['is_near_payday'].astype(int)

categorical_features = ['payment_method', 'issuing_bank', 'card_type', 'day_of_week']
numeric_features = [
    'amount', 'day_of_month', 'is_near_payday', 'retry_attempt_number',
    'consecutive_failure_count', 'customer_tenure_days', 'customer_ltv',
    'prior_failed_payments_count', 'prior_retry_success_rate', 'has_history',
    'hours_since_failure'
]

feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df['recovered'].astype(int)

for col in categorical_features:
    X[col] = X[col].fillna('unknown').astype(str)


# 3. Train/test split (stratified for class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Train recovery rate: {y_train.mean():.2%}, Test recovery rate: {y_test.mean():.2%}")


# 4. Baseline model — Logistic Regression
X_train_lr = pd.get_dummies(X_train, columns=categorical_features)
X_test_lr = pd.get_dummies(X_test, columns=categorical_features)
X_test_lr = X_test_lr.reindex(columns=X_train_lr.columns, fill_value=0)

scaler = StandardScaler()
X_train_lr_scaled = scaler.fit_transform(X_train_lr)
X_test_lr_scaled = scaler.transform(X_test_lr)

baseline_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
baseline_model.fit(X_train_lr_scaled, y_train)

baseline_preds = baseline_model.predict(X_test_lr_scaled)
baseline_probs = baseline_model.predict_proba(X_test_lr_scaled)[:, 1]

print("\n=== Baseline: Logistic Regression ===")
print(classification_report(y_test, baseline_preds, target_names=['not_recovered', 'recovered']))
print(f"ROC-AUC: {roc_auc_score(y_test, baseline_probs):.3f}")


# 5. Primary model — CatBoost (native categorical handling)
train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)

catboost_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    class_weights=[1, (1 - y_train.mean()) / y_train.mean()],  # handle imbalance
    random_seed=42,
    verbose=100
)

catboost_model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=30)

cb_preds = catboost_model.predict(X_test)
cb_probs = catboost_model.predict_proba(X_test)[:, 1]

print("\n=== Primary: CatBoost ===")
print(classification_report(y_test, cb_preds, target_names=['not_recovered', 'recovered']))
print(f"ROC-AUC: {roc_auc_score(y_test, cb_probs):.3f}")
print("Confusion matrix:\n", confusion_matrix(y_test, cb_preds))


# 6. Feature importance (explainability)
importances = catboost_model.get_feature_importance(train_pool)
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\n=== Feature Importance ===")
print(feature_importance_df.to_string(index=False))

# 7. Business-relevant evaluation: net recovered value-
# Assume a flat cost per retry attempt (tune this to your real SMS/gateway cost)
INTERVENTION_COST_PER_RETRY = 2.0  # ₹2 per retry attempt, adjust to real cost

eval_df = X_test.copy()
eval_df['amount'] = df.loc[X_test.index, 'amount'].values
eval_df['actual_recovered'] = y_test.values
eval_df['predicted_prob'] = cb_probs

# Simple policy: only retry if predicted_prob > 0.5
eval_df['would_retry'] = eval_df['predicted_prob'] > 0.5
eval_df['recovered_value'] = np.where(
    eval_df['would_retry'] & (eval_df['actual_recovered'] == 1),
    eval_df['amount'], 0
)
eval_df['cost'] = np.where(eval_df['would_retry'], INTERVENTION_COST_PER_RETRY, 0)
net_value = eval_df['recovered_value'].sum() - eval_df['cost'].sum()

print(f"\n=== Net Recovered Value (test set) ===")
print(f"Gross recovered: ₹{eval_df['recovered_value'].sum():,.2f}")
print(f"Total intervention cost: ₹{eval_df['cost'].sum():,.2f}")
print(f"Net recovered value: ₹{net_value:,.2f}")


# 8. Save the frozen model + preprocessing artifacts
catboost_model.save_model('recovery_model_catboost.cbm')
joblib.dump(baseline_model, 'recovery_model_logreg_baseline.pkl')
joblib.dump(scaler, 'logreg_scaler.pkl')
joblib.dump(list(X_train_lr.columns), 'logreg_columns.pkl')
feature_importance_df.to_csv('feature_importance.csv', index=False)

print("\nModels and artifacts saved:")
print("- recovery_model_catboost.cbm (primary, use this in your API)")
print("- recovery_model_logreg_baseline.pkl (baseline, for comparison)")
print("- feature_importance.csv")

Rows in scope: 1539
recovered
False    0.689409
True     0.310591
Name: proportion, dtype: float64
Train size: 1231, Test size: 308
Train recovery rate: 31.03%, Test recovery rate: 31.17%

=== Baseline: Logistic Regression ===
               precision    recall  f1-score   support

not_recovered       0.78      0.63      0.69       212
    recovered       0.42      0.60      0.50        96

     accuracy                           0.62       308
    macro avg       0.60      0.62      0.60       308
 weighted avg       0.67      0.62      0.63       308

ROC-AUC: 0.651
0:	test: 0.6433029	best: 0.6433029 (0)	total: 92.1ms	remaining: 46s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.6591735456
bestIteration = 6

Shrink model to first 7 iterations.

=== Primary: CatBoost ===
               precision    recall  f1-score   support

not_recovered       0.79      0.63      0.70       212
    recovered       0.43      0.62      0.51        96

     accuracy                

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from catboost import CatBoostClassifier, Pool
import joblib

# ---------------------------------------------------------
# 1. Load and scope data
# ---------------------------------------------------------
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/revenue_recovery_dataset.csv')
df = df[(df['failed_payment'] == True) & (df['decline_reason'] == 'insufficient_funds')].copy()
print(f"Rows in scope: {len(df)}")
print(df['recovered'].value_counts(normalize=True))

# ---------------------------------------------------------
# 2. Quick sanity check: does the bank signal even exist in the raw data?
# ---------------------------------------------------------
print("\n=== Raw recovery rate by issuing bank (before any model) ===")
bank_recovery = df.groupby('issuing_bank')['recovered'].agg(['mean', 'count'])
bank_recovery.columns = ['recovery_rate', 'n_transactions']
print(bank_recovery.sort_values('recovery_rate', ascending=False))

print("\n=== Raw recovery rate by day_of_week ===")
print(df.groupby('day_of_week')['recovered'].agg(['mean', 'count']))

print("\n=== Raw recovery rate: is_near_payday ===")
print(df.groupby('is_near_payday')['recovered'].agg(['mean', 'count']))

# ---------------------------------------------------------
# 3. Feature engineering
# ---------------------------------------------------------
df['prior_retry_success_rate'] = df['prior_retry_success_rate'].fillna(-1)
df['has_history'] = (df['prior_retry_success_rate'] != -1).astype(int)
df['is_near_payday'] = df['is_near_payday'].astype(int)

categorical_features = ['payment_method', 'issuing_bank', 'card_type', 'day_of_week']
numeric_features = [
    'amount', 'day_of_month', 'is_near_payday', 'retry_attempt_number',
    'consecutive_failure_count', 'customer_tenure_days', 'customer_ltv',
    'prior_failed_payments_count', 'prior_retry_success_rate', 'has_history',
    'hours_since_failure'
]
feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df['recovered'].astype(int)

for col in categorical_features:
    X[col] = X[col].fillna('unknown').astype(str)

# ---------------------------------------------------------
# 4. Cross-validated CatBoost (fixes the "stopped at iteration 6" problem)
# ---------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = []
all_importances = []

print("\n=== Cross-Validated CatBoost ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=categorical_features)
    val_pool = Pool(X_val, y_val, cat_features=categorical_features)

    model = CatBoostClassifier(
        iterations=300, learning_rate=0.03, depth=4,
        l2_leaf_reg=5,
        loss_function='Logloss', eval_metric='AUC',
        class_weights=[1, (1 - y_tr.mean()) / y_tr.mean()],
        random_seed=42, verbose=False
    )
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
    best_auc = model.get_best_score()['validation']['AUC']
    auc_scores.append(best_auc)
    all_importances.append(model.get_feature_importance(train_pool))
    print(f"Fold {fold}: AUC = {best_auc:.3f}, best iteration = {model.get_best_iteration()}")

print(f"\nMean AUC across folds: {np.mean(auc_scores):.3f} (+/- {np.std(auc_scores):.3f})")

# Average feature importance across folds — more stable than a single split
avg_importance = np.mean(all_importances, axis=0)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': avg_importance
}).sort_values('importance', ascending=False)
print("\n=== Averaged Feature Importance (across 5 folds) ===")
print(importance_df.to_string(index=False))

# ---------------------------------------------------------
# 5. Train final model on a proper train/test split for threshold tuning
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)

final_model = CatBoostClassifier(
    iterations=300, learning_rate=0.03, depth=4, l2_leaf_reg=5,
    loss_function='Logloss', eval_metric='AUC',
    class_weights=[1, (1 - y_train.mean()) / y_train.mean()],
    random_seed=42, verbose=False
)
final_model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=50)

cb_probs = final_model.predict_proba(X_test)[:, 1]
cb_preds_default = (cb_probs > 0.5).astype(int)

print("\n=== Final Model @ default 0.5 threshold ===")
print(classification_report(y_test, cb_preds_default, target_names=['not_recovered', 'recovered']))
print(f"ROC-AUC: {roc_auc_score(y_test, cb_probs):.3f}")

# ---------------------------------------------------------
# 6. Threshold sweep to maximize NET RECOVERED VALUE, not accuracy
# ---------------------------------------------------------
INTERVENTION_COST_PER_RETRY = 2.0  # replace with your real per-attempt cost

amounts_test = df.loc[X_test.index, 'amount'].values
thresholds = np.arange(0.10, 0.90, 0.05)
sweep_results = []

for t in thresholds:
    would_retry = cb_probs > t
    recovered_value = np.where(would_retry & (y_test.values == 1), amounts_test, 0).sum()
    cost = would_retry.sum() * INTERVENTION_COST_PER_RETRY
    net = recovered_value - cost
    sweep_results.append({
        'threshold': round(t, 2),
        'n_retries': int(would_retry.sum()),
        'gross_recovered': round(recovered_value, 2),
        'cost': round(cost, 2),
        'net_value': round(net, 2)
    })

sweep_df = pd.DataFrame(sweep_results).sort_values('net_value', ascending=False)
print("\n=== Threshold sweep (sorted by net value) ===")
print(sweep_df.to_string(index=False))

best_threshold = sweep_df.iloc[0]['threshold']
print(f"\nBest threshold for net value: {best_threshold}")

# ---------------------------------------------------------
# 7. Final evaluation at the best threshold
# ---------------------------------------------------------
cb_preds_best = (cb_probs > best_threshold).astype(int)
print(f"\n=== Final Model @ optimal threshold ({best_threshold}) ===")
print(classification_report(y_test, cb_preds_best, target_names=['not_recovered', 'recovered']))
print("Confusion matrix:\n", confusion_matrix(y_test, cb_preds_best))

# ---------------------------------------------------------
# 8. Save frozen model + metadata
# ---------------------------------------------------------
final_model.save_model('recovery_model_catboost.cbm')
joblib.dump({
    'best_threshold': best_threshold,
    'feature_cols': feature_cols,
    'categorical_features': categorical_features,
    'intervention_cost': INTERVENTION_COST_PER_RETRY
}, 'model_metadata.pkl')
importance_df.to_csv('feature_importance_cv.csv', index=False)
sweep_df.to_csv('threshold_sweep_results.csv', index=False)

print("\nSaved: recovery_model_catboost.cbm, model_metadata.pkl, feature_importance_cv.csv, threshold_sweep_results.csv")

Rows in scope: 1539
recovered
False    0.689409
True     0.310591
Name: proportion, dtype: float64

=== Raw recovery rate by issuing bank (before any model) ===
              recovery_rate  n_transactions
issuing_bank                               
HDFC               0.413043             138
ICICI              0.319018             163
Kotak              0.318471             157
SBI                0.296053             152
Axis               0.278912             147

=== Raw recovery rate by day_of_week ===
                 mean  count
day_of_week                 
Friday       0.291304    230
Monday       0.355556    225
Saturday     0.323276    232
Sunday       0.233918    171
Thursday     0.304721    233
Tuesday      0.322581    217
Wednesday    0.324675    231

=== Raw recovery rate: is_near_payday ===
                    mean  count
is_near_payday                 
False           0.229508    976
True            0.451155    563

=== Cross-Validated CatBoost ===
Fold 0: AUC = 0.727, be

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from catboost import CatBoostClassifier, Pool
import joblib

# ---------------------------------------------------------
# 1. Load and scope data
# ---------------------------------------------------------
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/revenue_recovery_dataset.csv')
df = df[(df['failed_payment'] == True) & (df['decline_reason'] == 'insufficient_funds')].copy()
print(f"Rows in scope: {len(df)}")
print(df['recovered'].value_counts(normalize=True))

# ---------------------------------------------------------
# 2. Feature engineering
# ---------------------------------------------------------
df['prior_retry_success_rate'] = df['prior_retry_success_rate'].fillna(-1)
df['has_history'] = (df['prior_retry_success_rate'] != -1).astype(int)
df['is_near_payday'] = df['is_near_payday'].astype(int)

categorical_features = ['payment_method', 'issuing_bank', 'card_type', 'day_of_week']
numeric_features = [
    'amount', 'day_of_month', 'is_near_payday', 'retry_attempt_number',
    'consecutive_failure_count', 'customer_tenure_days', 'customer_ltv',
    'prior_failed_payments_count', 'prior_retry_success_rate', 'has_history',
    'hours_since_failure'
]
feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df['recovered'].astype(int)

for col in categorical_features:
    X[col] = X[col].fillna('unknown').astype(str)

# ---------------------------------------------------------
# 3. Cross-validated CatBoost — FIX: more patience before stopping
# ---------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = []
all_importances = []

print("\n=== Cross-Validated CatBoost ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=categorical_features)
    val_pool = Pool(X_val, y_val, cat_features=categorical_features)

    model = CatBoostClassifier(
        iterations=500, learning_rate=0.03, depth=4,
        l2_leaf_reg=5,
        loss_function='Logloss', eval_metric='AUC',
        class_weights=[1, (1 - y_tr.mean()) / y_tr.mean()],
        random_seed=42, verbose=False,
        od_type='Iter', od_wait=100   # FIX: more patience before early stopping
    )
    model.fit(train_pool, eval_set=val_pool)
    best_auc = model.get_best_score()['validation']['AUC']
    auc_scores.append(best_auc)
    all_importances.append(model.get_feature_importance(train_pool))
    print(f"Fold {fold}: AUC = {best_auc:.3f}, best iteration = {model.get_best_iteration()}")

print(f"\nMean AUC across folds: {np.mean(auc_scores):.3f} (+/- {np.std(auc_scores):.3f})")

avg_importance = np.mean(all_importances, axis=0)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': avg_importance
}).sort_values('importance', ascending=False)
print("\n=== Averaged Feature Importance (across 5 folds) ===")
print(importance_df.to_string(index=False))

# ---------------------------------------------------------
# 4. Train final model on a held-out test split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)

final_model = CatBoostClassifier(
    iterations=500, learning_rate=0.03, depth=4, l2_leaf_reg=5,
    loss_function='Logloss', eval_metric='AUC',
    class_weights=[1, (1 - y_train.mean()) / y_train.mean()],
    random_seed=42, verbose=False,
    od_type='Iter', od_wait=100
)
final_model.fit(train_pool, eval_set=test_pool)
print(f"\nFinal model best iteration: {final_model.get_best_iteration()}")

cb_probs = final_model.predict_proba(X_test)[:, 1]
cb_preds_default = (cb_probs > 0.5).astype(int)

print("\n=== Final Model @ default 0.5 threshold ===")
print(classification_report(y_test, cb_preds_default, target_names=['not_recovered', 'recovered'], zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, cb_probs):.3f}")

# ---------------------------------------------------------
# 5. Threshold sweep — FIX: realistic cost model (bounce penalty on FAILED retries only)
# ---------------------------------------------------------
BOUNCE_PENALTY_PER_FAILED_RETRY = 250.0  # real NPCI OC-136-style cost, not a flat per-attempt fee

amounts_test = df.loc[X_test.index, 'amount'].values
thresholds = np.arange(0.10, 0.95, 0.05)
sweep_results = []

for t in thresholds:
    would_retry = cb_probs > t
    succeeded = would_retry & (y_test.values == 1)
    failed = would_retry & (y_test.values == 0)

    recovered_value = amounts_test[succeeded].sum()
    cost = failed.sum() * BOUNCE_PENALTY_PER_FAILED_RETRY
    net = recovered_value - cost

    sweep_results.append({
        'threshold': round(t, 2),
        'n_retries': int(would_retry.sum()),
        'n_succeeded': int(succeeded.sum()),
        'n_failed_and_penalized': int(failed.sum()),
        'gross_recovered': round(recovered_value, 2),
        'penalty_cost': round(cost, 2),
        'net_value': round(net, 2)
    })

sweep_df = pd.DataFrame(sweep_results).sort_values('net_value', ascending=False)
print("\n=== Threshold sweep (realistic bounce-penalty cost model) ===")
print(sweep_df.to_string(index=False))

best_threshold = sweep_df.iloc[0]['threshold']
print(f"\nBest threshold for net value: {best_threshold}")

# ---------------------------------------------------------
# 6. Final evaluation at the best threshold
# ---------------------------------------------------------
cb_preds_best = (cb_probs > best_threshold).astype(int)
print(f"\n=== Final Model @ optimal threshold ({best_threshold}) ===")
print(classification_report(y_test, cb_preds_best, target_names=['not_recovered', 'recovered'], zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, cb_preds_best))

# ---------------------------------------------------------
# 7. Save frozen model + metadata
# ---------------------------------------------------------
final_model.save_model('recovery_model_catboost.cbm')
joblib.dump({
    'best_threshold': best_threshold,
    'feature_cols': feature_cols,
    'categorical_features': categorical_features,
    'bounce_penalty': BOUNCE_PENALTY_PER_FAILED_RETRY
}, 'model_metadata.pkl')
importance_df.to_csv('feature_importance_cv.csv', index=False)
sweep_df.to_csv('threshold_sweep_results.csv', index=False)

print("\nSaved: recovery_model_catboost.cbm, model_metadata.pkl, feature_importance_cv.csv, threshold_sweep_results.csv")

Rows in scope: 1539
recovered
False    0.689409
True     0.310591
Name: proportion, dtype: float64

=== Cross-Validated CatBoost ===
Fold 0: AUC = 0.733, best iteration = 174
Fold 1: AUC = 0.622, best iteration = 274
Fold 2: AUC = 0.693, best iteration = 17
Fold 3: AUC = 0.696, best iteration = 5
Fold 4: AUC = 0.711, best iteration = 2

Mean AUC across folds: 0.691 (+/- 0.037)

=== Averaged Feature Importance (across 5 folds) ===
                    feature  importance
             is_near_payday   40.106038
   prior_retry_success_rate   18.494217
  consecutive_failure_count   10.483274
       retry_attempt_number    7.800699
               day_of_month    4.069064
               customer_ltv    3.465249
        hours_since_failure    3.164406
       customer_tenure_days    3.042197
                     amount    2.969863
             payment_method    1.845187
                day_of_week    1.236243
prior_failed_payments_count    1.151943
               issuing_bank    0.796451
      

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, roc_auc_score
from catboost import CatBoostClassifier, Pool
import joblib

# ---------------------------------------------------------
# 1. Load and scope data
# ---------------------------------------------------------
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/revenue_recovery_dataset.csv')
df = df[(df['failed_payment'] == True) & (df['decline_reason'] == 'insufficient_funds')].copy()
print(f"Rows in scope: {len(df)}")

# ---------------------------------------------------------
# 2. Feature engineering
# ---------------------------------------------------------
df['prior_retry_success_rate'] = df['prior_retry_success_rate'].fillna(-1)
df['has_history'] = (df['prior_retry_success_rate'] != -1).astype(int)
df['is_near_payday'] = df['is_near_payday'].astype(int)

categorical_features = ['payment_method', 'issuing_bank', 'card_type', 'day_of_week']
numeric_features = [
    'amount', 'day_of_month', 'is_near_payday', 'retry_attempt_number',
    'consecutive_failure_count', 'customer_tenure_days', 'customer_ltv',
    'prior_failed_payments_count', 'prior_retry_success_rate', 'has_history',
    'hours_since_failure'
]
feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df['recovered'].astype(int)

for col in categorical_features:
    X[col] = X[col].fillna('unknown').astype(str)

# ---------------------------------------------------------
# 3. THREE-way split: train / calibration / test
#    (calibration needs its own held-out data, separate from both
#     training and final evaluation)
# ---------------------------------------------------------
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_calib, y_train, y_calib = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)  # 0.25 of the remaining 80% = 20% of total, so ~60/20/20 train/calib/test

print(f"Train: {len(X_train)}, Calib: {len(X_calib)}, Test: {len(X_test)}")
amounts_test = df.loc[X_test.index, 'amount'].values

# ---------------------------------------------------------
# 4. Train the base model on the TRAIN split only
# ---------------------------------------------------------
base_model = CatBoostClassifier(
    iterations=500, learning_rate=0.03, depth=4, l2_leaf_reg=5,
    loss_function='Logloss', eval_metric='AUC',
    random_seed=42, verbose=False,
    od_type='Iter', od_wait=100,
    cat_features=categorical_features
)

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)
base_model.fit(train_pool, eval_set=test_pool)
print(f"Best iteration: {base_model.get_best_iteration()}")

raw_probs = base_model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC (uncalibrated): {roc_auc_score(y_test, raw_probs):.3f}")

# ---------------------------------------------------------
# 5. Calibrate using cv="prefit" — no cloning needed, fixes the RuntimeError
# ---------------------------------------------------------
calibrated_model = CalibratedClassifierCV(base_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_calib, y_calib)   # calibrate on the separate calibration split
calibrated_probs = calibrated_model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC (calibrated): {roc_auc_score(y_test, calibrated_probs):.3f}")

# ---------------------------------------------------------
# 6. Calibration check: does predicted prob match real outcome rate?
# ---------------------------------------------------------
calib_df = pd.DataFrame({'predicted_prob': calibrated_probs, 'actual': y_test.values})
calib_df['bucket'] = pd.cut(calib_df['predicted_prob'], bins=np.arange(0, 1.1, 0.1))
print("\n=== Calibration check (predicted vs actual, by bucket) ===")
print(calib_df.groupby('bucket', observed=True)['actual'].agg(['mean', 'count']))

# ---------------------------------------------------------
# 7. Per-transaction expected-value decision policy
# ---------------------------------------------------------
BOUNCE_PENALTY_PER_FAILED_RETRY = 250.0

expected_gain = calibrated_probs * amounts_test
expected_loss = (1 - calibrated_probs) * BOUNCE_PENALTY_PER_FAILED_RETRY
would_retry = expected_gain > expected_loss

succeeded = would_retry & (y_test.values == 1)
failed = would_retry & (y_test.values == 0)

recovered_value = amounts_test[succeeded].sum()
cost = failed.sum() * BOUNCE_PENALTY_PER_FAILED_RETRY
net = recovered_value - cost

print(f"\n=== Expected-Value Policy (calibrated, amount-aware) ===")
print(f"Retries attempted: {would_retry.sum()} out of {len(y_test)}")
print(f"Succeeded: {succeeded.sum()}, Failed & penalized: {failed.sum()}")
print(f"Gross recovered: ₹{recovered_value:,.2f}")
print(f"Penalty cost: ₹{cost:,.2f}")
print(f"Net recovered value: ₹{net:,.2f}")

print("\n=== Classification report under this policy ===")
print(classification_report(y_test, would_retry.astype(int),
      target_names=['not_recovered', 'recovered'], zero_division=0))

# ---------------------------------------------------------
# 8. Compare against baselines
# ---------------------------------------------------------
retry_all_value = amounts_test[y_test.values == 1].sum() - (y_test.values == 0).sum() * BOUNCE_PENALTY_PER_FAILED_RETRY
retry_none_value = 0.0

print(f"\n=== Baseline comparison ===")
print(f"Retry everyone:  net value = ₹{retry_all_value:,.2f}")
print(f"Retry no one:    net value = ₹{retry_none_value:,.2f}")
print(f"Expected-value policy: net value = ₹{net:,.2f}")

# ---------------------------------------------------------
# 9. Save everything
# ---------------------------------------------------------
base_model.save_model('recovery_model_catboost.cbm')
joblib.dump(calibrated_model, 'recovery_model_calibrated.pkl')
joblib.dump({
    'feature_cols': feature_cols,
    'categorical_features': categorical_features,
    'bounce_penalty': BOUNCE_PENALTY_PER_FAILED_RETRY
}, 'model_metadata.pkl')
calib_df.to_csv('calibration_check.csv', index=False)

print("\nSaved: recovery_model_catboost.cbm, recovery_model_calibrated.pkl, model_metadata.pkl, calibration_check.csv")

Rows in scope: 1539
Train: 923, Calib: 308, Test: 308
Best iteration: 241
ROC-AUC (uncalibrated): 0.660
ROC-AUC (calibrated): 0.665

=== Calibration check (predicted vs actual, by bucket) ===
                mean  count
bucket                     
(0.0, 0.1]  0.181818     33
(0.1, 0.2]  0.166667     54
(0.2, 0.3]  0.217391     69
(0.3, 0.4]  0.200000     10
(0.4, 0.5]  0.433333     60
(0.5, 0.6]  0.463768     69
(0.6, 0.7]  0.750000      4
(0.7, 0.8]  1.000000      1

=== Expected-Value Policy (calibrated, amount-aware) ===
Retries attempted: 218 out of 308
Succeeded: 78, Failed & penalized: 140
Gross recovered: ₹125,782.77
Penalty cost: ₹35,000.00
Net recovered value: ₹90,782.77

=== Classification report under this policy ===
               precision    recall  f1-score   support

not_recovered       0.80      0.34      0.48       212
    recovered       0.36      0.81      0.50        96

     accuracy                           0.49       308
    macro avg       0.58      0.58      

/usr/local/lib/python3.13/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [8]:
# ---------------------------------------------------------
# Rank transactions by expected value under a REAL capacity constraint
# ---------------------------------------------------------
CAPACITY_PCT = 0.05  # e.g. 5% review/outreach capacity, matching the reference project's cap
CAPACITY = max(1, int(CAPACITY_PCT * len(X_test)))

expected_value = calibrated_probs * amounts_test
rank_df = pd.DataFrame({
    'expected_value': expected_value,
    'predicted_prob': calibrated_probs,
    'amount': amounts_test,
    'actual': y_test.values
}).sort_values('expected_value', ascending=False).reset_index(drop=True)

top_n = rank_df.head(CAPACITY)
baseline_rate = y_test.mean()
top_n_rate = top_n['actual'].mean()

print(f"=== Capacity-constrained prioritization (top {CAPACITY} of {len(X_test)}, {CAPACITY_PCT:.0%} capacity) ===")
print(f"Overall test-set recovery rate:      {baseline_rate:.2%}")
print(f"Recovery rate within top {CAPACITY} picks:  {top_n_rate:.2%}")
print(f"Lift over baseline:                  {(top_n_rate / baseline_rate - 1):+.1%}")
print(f"Amount recovered from top {CAPACITY} picks:  ₹{top_n[top_n['actual']==1]['amount'].sum():,.2f}")
print(f"Amount recovered if picked randomly (expected): ₹{(baseline_rate * top_n['amount'].sum()):,.2f}")

# Sweep across a few capacity levels to show the full picture
print("\n=== Lift at different capacity levels ===")
for pct in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n = max(1, int(pct * len(X_test)))
    subset = rank_df.head(n)
    rate = subset['actual'].mean()
    print(f"Top {pct:.0%} ({n} cases): recovery rate = {rate:.2%}, lift = {(rate/baseline_rate - 1):+.1%}")

=== Capacity-constrained prioritization (top 15 of 308, 5% capacity) ===
Overall test-set recovery rate:      31.17%
Recovery rate within top 15 picks:  40.00%
Lift over baseline:                  +28.3%
Amount recovered from top 15 picks:  ₹16,041.59
Amount recovered if picked randomly (expected): ₹16,807.64

=== Lift at different capacity levels ===
Top 5% (15 cases): recovery rate = 40.00%, lift = +28.3%
Top 10% (30 cases): recovery rate = 40.00%, lift = +28.3%
Top 20% (61 cases): recovery rate = 42.62%, lift = +36.7%
Top 30% (92 cases): recovery rate = 40.22%, lift = +29.0%
Top 50% (154 cases): recovery rate = 39.61%, lift = +27.1%


In [9]:
from datetime import datetime, timedelta

def recommend_retry_schedule(model, calibration_model, transaction_features,
                               failure_date, categorical_features,
                               max_attempts=3, cooldown_hours=24,
                               candidate_window_days=10):
    """
    Given a failed transaction's features, scores multiple candidate retry dates
    and returns the best `max_attempts` days to use, respecting NPCI's 24h cooldown.
    """
    candidates = []
    current_date = failure_date + timedelta(hours=cooldown_hours)  # earliest legal retry

    for day_offset in range(candidate_window_days):
        candidate_date = current_date + timedelta(days=day_offset)
        day_of_month = candidate_date.day
        day_of_week = candidate_date.strftime('%A')
        is_near_payday = day_of_month <= 5 or day_of_month >= 25

        # build a feature row for this candidate date
        row = transaction_features.copy()
        row['day_of_month'] = day_of_month
        row['day_of_week'] = day_of_week
        row['is_near_payday'] = int(is_near_payday)

        row_df = pd.DataFrame([row])
        for col in categorical_features:
            row_df[col] = row_df[col].astype(str)

        prob = calibration_model.predict_proba(row_df)[:, 1][0]
        candidates.append({
            'date': candidate_date.strftime('%Y-%m-%d'),
            'day_of_week': day_of_week,
            'is_near_payday': is_near_payday,
            'predicted_prob': round(prob, 3)
        })

    candidates_df = pd.DataFrame(candidates).sort_values('predicted_prob', ascending=False)

    # respect the 24h cooldown between chosen attempts (here: at least 1 day apart,
    # since candidates are already daily; enforce max_attempts distinct spaced dates)
    chosen = []
    for _, row in candidates_df.iterrows():
        if len(chosen) >= max_attempts:
            break
        chosen.append(row)

    return pd.DataFrame(chosen).reset_index(drop=True)


# ---------------------------------------------------------
# Example usage: pick one failed test transaction and generate its schedule
# ---------------------------------------------------------
example_idx = X_test.index[0]
example_features = X_test.loc[example_idx].to_dict()
example_failure_date = datetime(2026, 6, 28)  # example: failed near month-end

schedule = recommend_retry_schedule(
    model=base_model,
    calibration_model=calibrated_model,
    transaction_features=example_features,
    failure_date=example_failure_date,
    categorical_features=categorical_features,
    max_attempts=3
)

print("=== Recommended retry schedule for this transaction ===")
print(schedule)

=== Recommended retry schedule for this transaction ===
         date day_of_week  is_near_payday  predicted_prob
0  2026-06-29      Monday            True           0.534
1  2026-06-30     Tuesday            True           0.534
2  2026-07-01   Wednesday            True           0.419


In [10]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

N_TRANSACTIONS = 20000       # up from 5,000
N_CUSTOMERS = 4000

# ---------------------------------------------------------
# 1. ID generators (unchanged)
# ---------------------------------------------------------
def generate_transaction_id():
    return ''.join(random.choices('0123456789', k=12))

def generate_customer_id():
    chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
    return 'cust_' + ''.join(random.choices(chars, k=14))

customer_pool = [generate_customer_id() for _ in range(N_CUSTOMERS)]
customer_profiles = {
    cid: {
        'tenure_days': int(np.random.exponential(scale=300)) + 1,
        'ltv': round(np.random.gamma(shape=2.0, scale=1500), 2),
        'true_recovery_tendency': np.random.beta(2, 3)
    }
    for cid in customer_pool
}

# ---------------------------------------------------------
# 2. BIN / bank pool (unchanged)
# ---------------------------------------------------------
bank_bins = {
    'HDFC': ['400012', '400013', '400014'],
    'ICICI': ['508227', '508228'],
    'SBI':   ['607331', '607332'],
    'Kotak': ['417612'],
    'Axis':  ['524104', '524105'],
}
bank_success_bias = {'HDFC': 0.62, 'ICICI': 0.55, 'SBI': 0.45, 'Kotak': 0.58, 'Axis': 0.50}

def random_bin_and_bank():
    bank = random.choice(list(bank_bins.keys()))
    bin_code = random.choice(bank_bins[bank])
    return bin_code, bank

# ---------------------------------------------------------
# 3. NEW: decline reasons, each with its own retry logic and base recovery rate
# ---------------------------------------------------------
DECLINE_REASONS = {
    'insufficient_funds': {
        'weight': 0.45,     # share of all failures
        'base_recovery': 0.30,
        'payday_sensitive': True,   # retrying near payday helps a lot
        'retryable': True
    },
    'server_timeout': {
        'weight': 0.20,
        'base_recovery': 0.70,      # transient issue, high recovery if retried soon
        'payday_sensitive': False,
        'retryable': True
    },
    'card_expired': {
        'weight': 0.15,
        'base_recovery': 0.05,      # retrying does nothing without a new card
        'payday_sensitive': False,
        'retryable': False          # needs payment-method update, not retry
    },
    'risk_hold': {
        'weight': 0.10,
        'base_recovery': 0.15,      # low, and retrying too fast makes it worse
        'payday_sensitive': False,
        'retryable': True           # but should be slow/limited
    },
    'card_limit_exceeded': {
        'weight': 0.10,
        'base_recovery': 0.40,      # resolves at billing-cycle reset, not payday
        'payday_sensitive': False,
        'retryable': True
    }
}
reason_names = list(DECLINE_REASONS.keys())
reason_weights = [DECLINE_REASONS[r]['weight'] for r in reason_names]

# ---------------------------------------------------------
# 4. Core generation loop
# ---------------------------------------------------------
rows = []
base_date = datetime(2026, 3, 1)  # wider date range for more variety

for i in range(N_TRANSACTIONS):
    txn_id = generate_transaction_id()
    customer_id = random.choice(customer_pool)
    profile = customer_profiles[customer_id]

    amount = round(np.random.gamma(shape=2.0, scale=800), 2)
    currency = 'INR'
    payment_method = random.choices(
        ['card', 'upi', 'netbanking', 'wallet'], weights=[0.45, 0.35, 0.15, 0.05]
    )[0]

    card_bin, issuing_bank, card_type = None, None, None
    if payment_method == 'card':
        card_bin, issuing_bank = random_bin_and_bank()
        card_type = random.choice(['debit', 'credit'])

    checkout_abandoned = np.random.rand() < 0.25
    reached_payment_page = np.random.rand() < 0.4 if checkout_abandoned else False

    payment_pending, pending_duration_minutes = False, None
    failed_payment = False
    decline_reason, decline_code = None, None
    failure_timestamp = None
    day_of_week, day_of_month, is_near_payday = None, None, None
    retry_attempt_number, consecutive_failure_count = None, None
    escalated_to_human = False
    retry_timestamp, hours_since_failure = None, None
    payment_status = 'success'

    if not checkout_abandoned:
        if payment_method in ['upi', 'netbanking']:
            payment_pending = np.random.rand() < 0.15
        if payment_pending:
            pending_duration_minutes = round(np.random.exponential(scale=25), 1)

        if not payment_pending:
            failed_payment = np.random.rand() < 0.45

        if failed_payment:
            decline_reason = random.choices(reason_names, weights=reason_weights)[0]
            reason_cfg = DECLINE_REASONS[decline_reason]
            decline_code = {
                'insufficient_funds': '51', 'server_timeout': 'TO1',
                'card_expired': '54', 'risk_hold': '59', 'card_limit_exceeded': '61'
            }[decline_reason]

            failure_dt = base_date + timedelta(days=int(np.random.rand() * 150),
                                                 hours=int(np.random.rand() * 24))
            failure_timestamp = failure_dt
            day_of_week = failure_dt.strftime('%A')
            day_of_month = failure_dt.day
            is_near_payday = day_of_month <= 5 or day_of_month >= 25

            if reason_cfg['retryable']:
                retry_attempt_number = np.random.choice([1, 2, 3], p=[0.5, 0.3, 0.2])
                consecutive_failure_count = retry_attempt_number

                hours_since_failure = round(
                    np.random.exponential(scale=12) if (reason_cfg['payday_sensitive'] and is_near_payday)
                    else np.random.exponential(scale=48), 1
                )
                retry_timestamp = failure_dt + timedelta(hours=hours_since_failure)

                base_prob = reason_cfg['base_recovery']
                if reason_cfg['payday_sensitive'] and is_near_payday:
                    base_prob += 0.20
                if issuing_bank:
                    base_prob += (bank_success_bias[issuing_bank] - 0.5) * 0.3
                base_prob += (profile['true_recovery_tendency'] - 0.5) * 0.25
                base_prob -= (retry_attempt_number - 1) * 0.05
                base_prob = min(max(base_prob, 0.02), 0.92)

                recovered = np.random.rand() < base_prob
            else:
                # not retryable (e.g. card_expired) — recovery only via method-update flow
                retry_attempt_number = 1
                consecutive_failure_count = 1
                recovered = np.random.rand() < reason_cfg['base_recovery']

            if consecutive_failure_count and consecutive_failure_count >= 3 and not recovered:
                escalated_to_human = True

            payment_status = 'recovered' if recovered else ('escalated' if escalated_to_human else 'unrecovered')
        else:
            payment_status = 'success'  # not abandoned, not pending, not failed
    else:
        recovered = np.random.rand() < (0.35 if reached_payment_page else 0.15)
        payment_status = 'recovered' if recovered else 'unrecovered'

    if payment_pending and not failed_payment:
        recovered = np.random.rand() < (0.55 if pending_duration_minutes < 20 else 0.30)
        payment_status = 'recovered' if recovered else 'unrecovered'

    prior_failed_payments_count = np.random.poisson(1.2)
    has_history = np.random.rand() > 0.30
    prior_retry_success_rate = round(profile['true_recovery_tendency'], 2) if has_history else np.nan

    rows.append({
        'transaction_id': txn_id, 'customer_id': customer_id, 'amount': amount, 'currency': currency,
        'payment_method': payment_method, 'card_bin': card_bin, 'issuing_bank': issuing_bank,
        'card_type': card_type, 'checkout_abandoned': checkout_abandoned,
        'reached_payment_page': reached_payment_page if checkout_abandoned else None,
        'payment_pending': payment_pending, 'pending_duration_minutes': pending_duration_minutes,
        'failed_payment': failed_payment, 'decline_reason': decline_reason, 'decline_code': decline_code,
        'failure_timestamp': failure_timestamp, 'day_of_week': day_of_week, 'day_of_month': day_of_month,
        'is_near_payday': is_near_payday, 'retry_attempt_number': retry_attempt_number,
        'consecutive_failure_count': consecutive_failure_count, 'escalated_to_human': escalated_to_human,
        'customer_tenure_days': profile['tenure_days'], 'customer_ltv': profile['ltv'],
        'prior_failed_payments_count': prior_failed_payments_count,
        'prior_retry_success_rate': prior_retry_success_rate,
        'retry_timestamp': retry_timestamp, 'hours_since_failure': hours_since_failure,
        'payment_status': payment_status,
    })

df = pd.DataFrame(rows)
df['recovered'] = df['payment_status'] == 'recovered'

print(f"Total rows: {len(df)}")
print(f"\nOverall payment_status distribution:\n{df['payment_status'].value_counts(normalize=True)}")
print(f"\nDecline reason distribution (failed payments only):\n{df[df['failed_payment']]['decline_reason'].value_counts()}")
print(f"\nRecovery rate by decline reason:")
print(df[df['failed_payment']].groupby('decline_reason')['recovered'].agg(['mean', 'count']))

df.to_csv('revenue_recovery_dataset_v2.csv', index=False)
print("\nSaved to revenue_recovery_dataset_v2.csv")

Total rows: 20000

Overall payment_status distribution:
payment_status
unrecovered    0.40095
success        0.38080
recovered      0.17975
escalated      0.03850
Name: proportion, dtype: float64

Decline reason distribution (failed payments only):
decline_reason
insufficient_funds     2868
server_timeout         1255
card_expired            934
risk_hold               620
card_limit_exceeded     607
Name: count, dtype: int64

Recovery rate by decline reason:
                         mean  count
decline_reason                      
card_expired         0.044968    934
card_limit_exceeded  0.321252    607
insufficient_funds   0.327406   2868
risk_hold            0.085484    620
server_timeout       0.640637   1255

Saved to revenue_recovery_dataset_v2.csv


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, roc_auc_score
from catboost import CatBoostClassifier, Pool
import joblib

# ---------------------------------------------------------
# 1. Load the FULL dataset — all failed payments, all decline reasons
# ---------------------------------------------------------
df = pd.read_csv('/content/revenue_recovery_dataset_v2.csv')
df = df[df['failed_payment'] == True].copy()
print(f"Rows in scope (all decline reasons): {len(df)}")
print(f"\nDecline reason counts:\n{df['decline_reason'].value_counts()}")
print(f"\nOverall recovery rate: {df['recovered'].mean():.2%}")
print(f"\nRecovery rate by decline reason:")
print(df.groupby('decline_reason')['recovered'].agg(['mean', 'count']))

# ---------------------------------------------------------
# 2. Feature engineering — decline_reason is now a real feature
# ---------------------------------------------------------
df['prior_retry_success_rate'] = df['prior_retry_success_rate'].fillna(-1)
df['has_history'] = (df['prior_retry_success_rate'] != -1).astype(int)
df['is_near_payday'] = df['is_near_payday'].fillna(False).astype(int)
df['hours_since_failure'] = df['hours_since_failure'].fillna(df['hours_since_failure'].median())

categorical_features = ['payment_method', 'issuing_bank', 'card_type', 'day_of_week', 'decline_reason']
numeric_features = [
    'amount', 'day_of_month', 'is_near_payday', 'retry_attempt_number',
    'consecutive_failure_count', 'customer_tenure_days', 'customer_ltv',
    'prior_failed_payments_count', 'prior_retry_success_rate', 'has_history',
    'hours_since_failure'
]
feature_cols = categorical_features + numeric_features
X = df[feature_cols].copy()
y = df['recovered'].astype(int)

for col in categorical_features:
    X[col] = X[col].fillna('unknown').astype(str)

# ---------------------------------------------------------
# 3. Three-way split: train / calibration / test
# ---------------------------------------------------------
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_calib, y_train, y_calib = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"\nTrain: {len(X_train)}, Calib: {len(X_calib)}, Test: {len(X_test)}")
amounts_test = df.loc[X_test.index, 'amount'].values
reasons_test = df.loc[X_test.index, 'decline_reason'].values

# ---------------------------------------------------------
# 4. Train CatBoost on the full multi-reason dataset
# ---------------------------------------------------------
base_model = CatBoostClassifier(
    iterations=800, learning_rate=0.03, depth=5, l2_leaf_reg=5,
    loss_function='Logloss', eval_metric='AUC',
    random_seed=42, verbose=False,
    od_type='Iter', od_wait=100,
    cat_features=categorical_features
)

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, y_test, cat_features=categorical_features)
base_model.fit(train_pool, eval_set=test_pool)
print(f"\nBest iteration: {base_model.get_best_iteration()}")

raw_probs = base_model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC (uncalibrated): {roc_auc_score(y_test, raw_probs):.3f}")

# ---------------------------------------------------------
# 5. Feature importance — check whether decline_reason now matters,
#    and whether issuing_bank finally shows real importance with more data
# ---------------------------------------------------------
importances = base_model.get_feature_importance(train_pool)
importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances}).sort_values('importance', ascending=False)
print("\n=== Feature Importance (full dataset) ===")
print(importance_df.to_string(index=False))

# ---------------------------------------------------------
# 6. Calibrate using cv='prefit' on the separate calibration split
# ---------------------------------------------------------
calibrated_model = CalibratedClassifierCV(base_model, method='isotonic', cv='prefit')
calibrated_model.fit(X_calib, y_calib)
calibrated_probs = calibrated_model.predict_proba(X_test)[:, 1]
print(f"\nROC-AUC (calibrated): {roc_auc_score(y_test, calibrated_probs):.3f}")

# ---------------------------------------------------------
# 7. Per-decline-reason performance — does the model correctly
#    treat card_expired differently from insufficient_funds?
# ---------------------------------------------------------
eval_df = pd.DataFrame({
    'decline_reason': reasons_test,
    'predicted_prob': calibrated_probs,
    'actual': y_test.values,
    'amount': amounts_test
})
print("\n=== Model behavior by decline reason ===")
print(eval_df.groupby('decline_reason').agg(
    avg_predicted_prob=('predicted_prob', 'mean'),
    actual_recovery_rate=('actual', 'mean'),
    count=('actual', 'count')
))

# ---------------------------------------------------------
# 8. Capacity-constrained ranking, on the full test set
# ---------------------------------------------------------
expected_value = calibrated_probs * amounts_test
rank_df = pd.DataFrame({
    'expected_value': expected_value, 'predicted_prob': calibrated_probs,
    'amount': amounts_test, 'actual': y_test.values, 'decline_reason': reasons_test
}).sort_values('expected_value', ascending=False).reset_index(drop=True)

baseline_rate = y_test.mean()
print(f"\n=== Lift at different capacity levels (full dataset, all reasons) ===")
for pct in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n = max(1, int(pct * len(X_test)))
    subset = rank_df.head(n)
    rate = subset['actual'].mean()
    amt_recovered = subset[subset['actual']==1]['amount'].sum()
    print(f"Top {pct:.0%} ({n} cases): recovery rate = {rate:.2%}, lift = {(rate/baseline_rate - 1):+.1%}, "
          f"₹ recovered = {amt_recovered:,.2f}")

# ---------------------------------------------------------
# 9. Save everything
# ---------------------------------------------------------
base_model.save_model('recovery_model_catboost_v2.cbm')
joblib.dump(calibrated_model, 'recovery_model_calibrated_v2.pkl')
joblib.dump({'feature_cols': feature_cols, 'categorical_features': categorical_features}, 'model_metadata_v2.pkl')
importance_df.to_csv('feature_importance_v2.csv', index=False)

print("\nSaved: recovery_model_catboost_v2.cbm, recovery_model_calibrated_v2.pkl, model_metadata_v2.pkl, feature_importance_v2.csv")

Rows in scope (all decline reasons): 6284

Decline reason counts:
decline_reason
insufficient_funds     2868
server_timeout         1255
card_expired            934
risk_hold               620
card_limit_exceeded     607
Name: count, dtype: int64

Overall recovery rate: 32.35%

Recovery rate by decline reason:
                         mean  count
decline_reason                      
card_expired         0.044968    934
card_limit_exceeded  0.321252    607
insufficient_funds   0.327406   2868
risk_hold            0.085484    620
server_timeout       0.640637   1255

Train: 3770, Calib: 1257, Test: 1257


/tmp/ipykernel_2517/3904632339.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_near_payday'] = df['is_near_payday'].fillna(False).astype(int)



Best iteration: 42
ROC-AUC (uncalibrated): 0.762

=== Feature Importance (full dataset) ===
                    feature  importance
             decline_reason   82.692893
             is_near_payday    7.793169
   prior_retry_success_rate    2.305685
               day_of_month    2.273338
        hours_since_failure    1.288356
       retry_attempt_number    0.993660
  consecutive_failure_count    0.636398
                has_history    0.584277
                     amount    0.422481
       customer_tenure_days    0.381275
               customer_ltv    0.345225
prior_failed_payments_count    0.200634
             payment_method    0.041371
               issuing_bank    0.041238
                  card_type    0.000000
                day_of_week    0.000000

ROC-AUC (calibrated): 0.762

=== Model behavior by decline reason ===
                     avg_predicted_prob  actual_recovery_rate  count
decline_reason                                                      
card_expired      

/usr/local/lib/python3.13/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [12]:
# ---------------------------------------------------------
# 8B. WITHIN-REASON TARGETING TEST
# Does the model actually rank customers within
# insufficient_funds, rather than simply ranking reasons?
# ---------------------------------------------------------

print("\n=== WITHIN-REASON TARGETING ===")

for reason in ['insufficient_funds',
               'server_timeout',
               'card_expired',
               'card_limit_exceeded']:

    subset = rank_df[
        rank_df['decline_reason'] == reason
    ].copy()

    if len(subset) < 20:
        print(f"\n{reason}: Not enough test cases ({len(subset)})")
        continue

    baseline = subset['actual'].mean()

    print(f"\n--- {reason} ---")
    print(f"Cases: {len(subset)}")
    print(f"Baseline recovery: {baseline:.2%}")

    for pct in [0.10, 0.20, 0.30, 0.50]:

        n = max(1, int(pct * len(subset)))

        top = subset.sort_values(
            'expected_value',
            ascending=False
        ).head(n)

        rate = top['actual'].mean()

        lift = (
            rate / baseline - 1
            if baseline > 0
            else np.nan
        )

        recovered_amount = top[
            top['actual'] == 1
        ]['amount'].sum()

        print(
            f"Top {pct:.0%} ({n} cases): "
            f"recovery={rate:.2%}, "
            f"lift={lift:+.1%}, "
            f"₹ recovered={recovered_amount:,.2f}"
        )


=== WITHIN-REASON TARGETING ===

--- insufficient_funds ---
Cases: 548
Baseline recovery: 32.12%
Top 10% (54 cases): recovery=37.04%, lift=+15.3%, ₹ recovered=72,529.29
Top 20% (109 cases): recovery=41.28%, lift=+28.5%, ₹ recovered=129,351.30
Top 30% (164 cases): recovery=37.80%, lift=+17.7%, ₹ recovered=165,902.58
Top 50% (274 cases): recovery=33.94%, lift=+5.7%, ₹ recovered=220,847.19

--- server_timeout ---
Cases: 271
Baseline recovery: 63.47%
Top 10% (27 cases): recovery=62.96%, lift=-0.8%, ₹ recovered=62,791.71
Top 20% (54 cases): recovery=62.96%, lift=-0.8%, ₹ recovered=108,790.35
Top 30% (81 cases): recovery=64.20%, lift=+1.1%, ₹ recovered=149,231.91
Top 50% (135 cases): recovery=63.70%, lift=+0.4%, ₹ recovered=204,103.45

--- card_expired ---
Cases: 193
Baseline recovery: 5.70%
Top 10% (19 cases): recovery=10.53%, lift=+84.7%, ₹ recovered=8,262.60
Top 20% (38 cases): recovery=10.53%, lift=+84.7%, ₹ recovered=13,326.07
Top 30% (57 cases): recovery=8.77%, lift=+53.9%, ₹ recovere

In [14]:
# =========================================================
# INSUFFICIENT FUNDS — FEATURE ANALYSIS
# =========================================================

print("\n=== INSUFFICIENT FUNDS ANALYSIS ===")

# Get test-set rows that are insufficient_funds
insuff_indices = X_test.index[
    df.loc[X_test.index, 'decline_reason'] == 'insufficient_funds'
]

# Original dataframe rows
insuff = df.loc[insuff_indices].copy()

# Add model predictions using the SAME indices
# Create a mapping from test index -> calibrated probability
prob_map = pd.Series(
    calibrated_probs,
    index=X_test.index
)

insuff['predicted_prob'] = prob_map.loc[insuff.index]

print(f"\nTotal insufficient_funds test cases: {len(insuff)}")


# ---------------------------------------------------------
# 1. PAYDAY EFFECT
# ---------------------------------------------------------

print("\n=== Recovery by Payday ===")

payday_analysis = insuff.groupby(
    'is_near_payday'
)['recovered'].agg(
    recovery_rate='mean',
    cases='count',
    recovered='sum'
)

print(payday_analysis)

print("\n")


# ---------------------------------------------------------
# 2. RETRY ATTEMPT EFFECT
# ---------------------------------------------------------

print("=== Recovery by Retry Attempt ===")

retry_analysis = insuff.groupby(
    'retry_attempt_number'
)['recovered'].agg(
    recovery_rate='mean',
    cases='count',
    recovered='sum'
)

print(retry_analysis)

print("\n")


# ---------------------------------------------------------
# 3. PRIOR RETRY SUCCESS RATE
# ---------------------------------------------------------

print("=== Recovery by Prior Retry Success Rate ===")

# Remove missing/no-history values for this analysis
history = insuff[
    insuff['prior_retry_success_rate'] >= 0
].copy()

if len(history) > 0:

    history['history_bucket'] = pd.cut(
        history['prior_retry_success_rate'],
        bins=[-0.01, 0.25, 0.50, 0.75, 1.00],
        labels=[
            '0-25%',
            '25-50%',
            '50-75%',
            '75-100%'
        ]
    )

    history_analysis = history.groupby(
        'history_bucket',
        observed=True
    )['recovered'].agg(
        recovery_rate='mean',
        cases='count',
        recovered='sum'
    )

    print(history_analysis)

else:
    print("No historical retry data available.")


print("\n")


# ---------------------------------------------------------
# 4. DAY OF MONTH
# ---------------------------------------------------------

print("=== Recovery by Day of Month ===")

day_analysis = insuff.groupby(
    'day_of_month'
)['recovered'].agg(
    recovery_rate='mean',
    cases='count',
    recovered='sum'
)

print(day_analysis)

print("\n")


# ---------------------------------------------------------
# 5. CUSTOMER HISTORY
# ---------------------------------------------------------

print("=== Recovery by Previous Failed Payments ===")

failure_analysis = insuff.groupby(
    'prior_failed_payments_count'
)['recovered'].agg(
    recovery_rate='mean',
    cases='count',
    recovered='sum'
)

print(failure_analysis)

print("\n")


# ---------------------------------------------------------
# 6. MODEL PROBABILITY DISTRIBUTION
# ---------------------------------------------------------

print("=== Predicted Probability Distribution ===")

print(
    insuff['predicted_prob'].describe()
)


=== INSUFFICIENT FUNDS ANALYSIS ===

Total insufficient_funds test cases: 548

=== Recovery by Payday ===
                recovery_rate  cases  recovered
is_near_payday                                 
0                    0.249284    349         87
1                    0.447236    199         89


=== Recovery by Retry Attempt ===
                      recovery_rate  cases  recovered
retry_attempt_number                                 
1.0                        0.339921    253         86
2.0                        0.305732    157         48
3.0                        0.304348    138         42


=== Recovery by Prior Retry Success Rate ===
                recovery_rate  cases  recovered
history_bucket                                 
0-25%                0.268817     93         25
25-50%               0.337209    172         58
50-75%               0.384615    117         45
75-100%              0.333333     12          4


=== Recovery by Day of Month ===
              recovery_ra

In [15]:
# =========================================================
# COMPARE ML vs SIMPLE RULES
# =========================================================

print("\n" + "="*60)
print("ML vs SIMPLE TARGETING")
print("="*60)

insuff = df.loc[insuff_indices].copy()

# Add model probability
insuff['predicted_prob'] = prob_map.loc[insuff.index]

# ---------------------------------------------------------
# BASELINE
# ---------------------------------------------------------

baseline = insuff['recovered'].mean()

print(f"\nBaseline recovery: {baseline:.2%}")


# ---------------------------------------------------------
# RULE 1: NEAR PAYDAY
# ---------------------------------------------------------

payday_customers = insuff[
    insuff['is_near_payday'] == 1
]

print("\n=== RULE: NEAR PAYDAY ===")

print(
    f"Cases: {len(payday_customers)}"
)

print(
    f"Recovery: "
    f"{payday_customers['recovered'].mean():.2%}"
)

print(
    f"Lift: "
    f"{(payday_customers['recovered'].mean()/baseline - 1):+.2%}"
)


# ---------------------------------------------------------
# RULE 2: PRIOR RETRY SUCCESS >= 50%
# ---------------------------------------------------------

history_customers = insuff[
    insuff['prior_retry_success_rate'] >= 0.50
]

print("\n=== RULE: PRIOR RETRY SUCCESS >= 50% ===")

print(
    f"Cases: {len(history_customers)}"
)

print(
    f"Recovery: "
    f"{history_customers['recovered'].mean():.2%}"
)

print(
    f"Lift: "
    f"{(history_customers['recovered'].mean()/baseline - 1):+.2%}"
)


# ---------------------------------------------------------
# RULE 3: ML TOP 20%
# ---------------------------------------------------------

top20 = insuff.nlargest(
    int(len(insuff) * 0.20),
    'predicted_prob'
)

print("\n=== ML TOP 20% ===")

print(
    f"Cases: {len(top20)}"
)

print(
    f"Recovery: "
    f"{top20['recovered'].mean():.2%}"
)

print(
    f"Lift: "
    f"{(top20['recovered'].mean()/baseline - 1):+.2%}"
)


# ---------------------------------------------------------
# RULE 4: ML TOP 10%
# ---------------------------------------------------------

top10 = insuff.nlargest(
    int(len(insuff) * 0.10),
    'predicted_prob'
)

print("\n=== ML TOP 10% ===")

print(
    f"Cases: {len(top10)}"
)

print(
    f"Recovery: "
    f"{top10['recovered'].mean():.2%}"
)

print(
    f"Lift: "
    f"{(top10['recovered'].mean()/baseline - 1):+.2%}"
)


ML vs SIMPLE TARGETING

Baseline recovery: 32.12%

=== RULE: NEAR PAYDAY ===
Cases: 199
Recovery: 44.72%
Lift: +39.25%

=== RULE: PRIOR RETRY SUCCESS >= 50% ===
Cases: 136
Recovery: 37.50%
Lift: +16.76%

=== ML TOP 20% ===
Cases: 109
Recovery: 52.29%
Lift: +62.82%

=== ML TOP 10% ===
Cases: 54
Recovery: 46.30%
Lift: +44.15%


In [16]:
# =========================================================
# INSUFFICIENT FUNDS SPECIALIST MODEL
# =========================================================

insuff_df = df[
    (df['failed_payment'] == True) &
    (df['decline_reason'] == 'insufficient_funds')
].copy()

print("Cases:", len(insuff_df))
print(
    "Recovery rate:",
    insuff_df['recovered'].mean()
)

Cases: 2868
Recovery rate: 0.3274058577405858


In [17]:
# =========================================================
# SPECIALIST MODEL — INSUFFICIENT FUNDS ONLY
# =========================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier, Pool
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# DATA
# ---------------------------------------------------------

insuff_df = df[
    (df['failed_payment'] == True) &
    (df['decline_reason'] == 'insufficient_funds')
].copy()

print("Total insufficient_funds cases:", len(insuff_df))
print(
    "Overall recovery rate:",
    insuff_df['recovered'].mean()
)


# ---------------------------------------------------------
# FEATURES
# ---------------------------------------------------------

categorical_features_specialist = [
    'payment_method',
    'issuing_bank',
    'card_type',
    'day_of_week'
]

numeric_features_specialist = [
    'amount',
    'day_of_month',
    'is_near_payday',
    'retry_attempt_number',
    'consecutive_failure_count',
    'customer_tenure_days',
    'customer_ltv',
    'prior_failed_payments_count',
    'prior_retry_success_rate',
    'has_history',
    'hours_since_failure'
]

specialist_features = (
    categorical_features_specialist +
    numeric_features_specialist
)

X = insuff_df[specialist_features].copy()
y = insuff_df['recovered'].astype(int)


# ---------------------------------------------------------
# CLEAN
# ---------------------------------------------------------

for col in categorical_features_specialist:
    X[col] = X[col].fillna('unknown').astype(str)

X['prior_retry_success_rate'] = (
    X['prior_retry_success_rate'].fillna(-1)
)

X['has_history'] = (
    X['prior_retry_success_rate'] != -1
).astype(int)

X['is_near_payday'] = (
    X['is_near_payday']
    .fillna(False)
    .astype(int)
)

X['hours_since_failure'] = (
    X['hours_since_failure']
    .fillna(X['hours_since_failure'].median())
)


# ---------------------------------------------------------
# SPLIT
# ---------------------------------------------------------

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_calib, y_train, y_calib = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("\nSplit:")
print("Train:", len(X_train))
print("Calibration:", len(X_calib))
print("Test:", len(X_test))


# ---------------------------------------------------------
# MODEL
# ---------------------------------------------------------

model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.03,
    depth=5,
    l2_leaf_reg=5,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=False,
    od_type='Iter',
    od_wait=100,
    cat_features=categorical_features_specialist
)

train_pool = Pool(
    X_train,
    y_train,
    cat_features=categorical_features_specialist
)

calib_pool = Pool(
    X_calib,
    y_calib,
    cat_features=categorical_features_specialist
)

# IMPORTANT:
# Don't use test set for early stopping.
model.fit(
    train_pool,
    eval_set=calib_pool
)


# ---------------------------------------------------------
# CALIBRATION
# ---------------------------------------------------------

calibrated_model = CalibratedClassifierCV(
    model,
    method='isotonic',
    cv='prefit'
)

calibrated_model.fit(
    X_calib,
    y_calib
)


# ---------------------------------------------------------
# TEST
# ---------------------------------------------------------

test_probs = calibrated_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(
    y_test,
    test_probs
)

print("\n=== SPECIALIST MODEL ===")
print(f"Test AUC: {auc:.4f}")

Total insufficient_funds cases: 2868
Overall recovery rate: 0.3274058577405858

Split:
Train: 1720
Calibration: 574
Test: 574

=== SPECIALIST MODEL ===
Test AUC: 0.6132


/usr/local/lib/python3.13/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


In [18]:
# =========================================================
# SPECIALIST MODEL TARGETING
# =========================================================

specialist_results = pd.DataFrame({
    'probability': test_probs,
    'actual': y_test.values,
    'amount': insuff_df.loc[X_test.index, 'amount'].values
})

baseline = specialist_results['actual'].mean()

print("\n" + "="*60)
print("SPECIALIST MODEL TARGETING")
print("="*60)

print(
    f"\nTest baseline recovery: {baseline:.2%}"
)


# ---------------------------------------------------------
# TOP 5 / 10 / 20 / 30 / 50%
# ---------------------------------------------------------

for pct in [0.05, 0.10, 0.20, 0.30, 0.50]:

    n = max(
        1,
        int(len(specialist_results) * pct)
    )

    top = specialist_results.nlargest(
        n,
        'probability'
    )

    recovery = top['actual'].mean()

    lift = (
        recovery / baseline - 1
    )

    recovered_amount = (
        top.loc[
            top['actual'] == 1,
            'amount'
        ].sum()
    )

    print(
        f"\nTop {pct:.0%}"
    )

    print(
        f"Cases: {n}"
    )

    print(
        f"Recovery: {recovery:.2%}"
    )

    print(
        f"Lift: {lift:+.2%}"
    )

    print(
        f"Recovered ₹: "
        f"{recovered_amount:,.2f}"
    )


SPECIALIST MODEL TARGETING

Test baseline recovery: 32.75%

Top 5%
Cases: 28
Recovery: 57.14%
Lift: +74.47%
Recovered ₹: 20,301.49

Top 10%
Cases: 57
Recovery: 50.88%
Lift: +55.34%
Recovered ₹: 44,970.87

Top 20%
Cases: 114
Recovery: 43.86%
Lift: +33.91%
Recovered ₹: 81,372.34

Top 30%
Cases: 172
Recovery: 43.60%
Lift: +33.13%
Recovered ₹: 128,370.02

Top 50%
Cases: 287
Recovery: 41.11%
Lift: +25.53%
Recovered ₹: 202,640.41


In [19]:
# =========================================================
# SPECIALIST FEATURE IMPORTANCE
# =========================================================

importance = model.get_feature_importance(
    train_pool
)

importance_df = pd.DataFrame({
    'feature': specialist_features,
    'importance': importance
}).sort_values(
    'importance',
    ascending=False
)

print("\n=== SPECIALIST FEATURE IMPORTANCE ===")
print(importance_df.to_string(index=False))


=== SPECIALIST FEATURE IMPORTANCE ===
                    feature  importance
             is_near_payday   63.526222
   prior_retry_success_rate   10.159592
               day_of_month    6.310098
  consecutive_failure_count    4.679671
       retry_attempt_number    4.098766
       customer_tenure_days    3.430964
               customer_ltv    2.707559
             payment_method    2.222263
        hours_since_failure    1.492642
                  card_type    0.563426
                     amount    0.297780
                has_history    0.255606
prior_failed_payments_count    0.255411
                day_of_week    0.000000
               issuing_bank    0.000000


In [20]:
# =========================================================
# SPECIALIST MODEL — DECILE ANALYSIS
# =========================================================

results = specialist_results.copy()

results['decile'] = pd.qcut(
    results['probability'],
    q=10,
    labels=False,
    duplicates='drop'
)

decile_analysis = results.groupby(
    'decile',
    observed=True
).agg(
    cases=('actual', 'count'),
    recovery_rate=('actual', 'mean'),
    avg_probability=('probability', 'mean'),
    recovered_amount=('amount', lambda x: 0)
)

# Calculate actual recovered ₹ separately
decile_analysis['recovered_cases'] = (
    results.groupby(
        'decile',
        observed=True
    )['actual'].sum()
)

decile_analysis['lift'] = (
    decile_analysis['recovery_rate']
    / results['actual'].mean()
    - 1
)

print("\n=== SPECIALIST MODEL DECILE ANALYSIS ===")

print(
    decile_analysis
)


=== SPECIALIST MODEL DECILE ANALYSIS ===
        cases  recovery_rate  avg_probability  recovered_amount  \
decile                                                            
0          68       0.235294         0.202294                 0   
1          83       0.253012         0.236667                 0   
2          38       0.236842         0.280000                 0   
3          85       0.247059         0.285714                 0   
4         109       0.321101         0.293651                 0   
5         144       0.423611         0.445611                 0   
6          47       0.531915         0.562431                 0   

        recovered_cases      lift  
decile                             
0                    16 -0.281602  
1                    21 -0.227506  
2                     9 -0.276876  
3                    21 -0.245682  
4                    35 -0.019617  
5                    61  0.293366  
6                    25  0.624038  


In [21]:
decile_analysis = results.groupby(
    'decile',
    observed=True
).apply(
    lambda g: pd.Series({
        'cases': len(g),
        'recovery_rate': g['actual'].mean(),
        'avg_probability': g['probability'].mean(),
        'recovered_cases': g['actual'].sum(),
        'recovered_amount': g.loc[
            g['actual'] == 1,
            'amount'
        ].sum()
    }),
    include_groups=False
)

baseline = results['actual'].mean()

decile_analysis['lift'] = (
    decile_analysis['recovery_rate'] / baseline - 1
)

print(
    decile_analysis
)

        cases  recovery_rate  avg_probability  recovered_cases  \
decile                                                           
0        68.0       0.235294         0.202294             16.0   
1        83.0       0.253012         0.236667             21.0   
2        38.0       0.236842         0.280000              9.0   
3        85.0       0.247059         0.285714             21.0   
4       109.0       0.321101         0.293651             35.0   
5       144.0       0.423611         0.445611             61.0   
6        47.0       0.531915         0.562431             25.0   

        recovered_amount      lift  
decile                              
0               32980.07 -0.281602  
1               33904.82 -0.227506  
2               16361.14 -0.276876  
3               29366.89 -0.245682  
4               54827.17 -0.019617  
5              109476.85  0.293366  
6               40071.61  0.624038  


In [22]:
# =========================================================
# ABLATION STUDY
# =========================================================

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# PAYDAY ONLY
# ---------------------------------------------------------

X_payday = X[['is_near_payday']]

Xtr, Xte, ytr, yte = train_test_split(
    X_payday,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

payday_model = LogisticRegression()

payday_model.fit(Xtr, ytr)

payday_probs = payday_model.predict_proba(Xte)[:, 1]

payday_auc = roc_auc_score(
    yte,
    payday_probs
)

print("\n=== PAYDAY ONLY ===")
print("AUC:", payday_auc)


# ---------------------------------------------------------
# PAYDAY + HISTORY
# ---------------------------------------------------------

selected_features = [
    'is_near_payday',
    'prior_retry_success_rate',
    'day_of_month',
    'retry_attempt_number',
    'consecutive_failure_count'
]

X_small = X[selected_features]

Xtr, Xte, ytr, yte = train_test_split(
    X_small,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

small_model = CatBoostClassifier(
    iterations=500,
    depth=4,
    learning_rate=0.03,
    loss_function='Logloss',
    verbose=False,
    random_seed=42
)

small_model.fit(Xtr, ytr)

small_probs = small_model.predict_proba(Xte)[:, 1]

small_auc = roc_auc_score(
    yte,
    small_probs
)

print("\n=== PAYDAY + HISTORY ===")
print("AUC:", small_auc)


# ---------------------------------------------------------
# FULL SPECIALIST
# ---------------------------------------------------------

print("\n=== FULL SPECIALIST ===")
print("AUC:", auc)


=== PAYDAY ONLY ===
AUC: 0.5880140006614486

=== PAYDAY + HISTORY ===
AUC: 0.6070995480101422

=== FULL SPECIALIST ===
AUC: 0.6131628265902326
